# Experiment 6: Multilayer Perceptron for the XOR Problem
**Course:** CSAI4014 Algorithms for Intelligent Systems and Robotics  
**Student:** Aadi Chopra | SAP ID: 500120163 | Roll No: R2142231902 | Batch: B2 AIML  
**Repository:** https://github.com/aadichopra/aisr-lab-500120163  

## 1. Aim
To implement a 2-4-1 Multilayer Perceptron from scratch using NumPy, train it with backpropagation on the nonlinearly separable XOR problem, evaluate the effect of learning rates $\eta \in \{0.1, 0.5, 2.0\}$, and plot the nonlinear decision boundary.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('figs', exist_ok=True)

def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -25, 25)))
def sigmoid_deriv(a): return a * (1.0 - a)

def train_mlp(lr, epochs=10000, seed=42):
    rng = np.random.default_rng(seed)
    X = np.array([[0,0], [0,1], [1,0], [1,1]])
    y = np.array([[0], [1], [1], [0]])
    W1 = rng.uniform(-1.0, 1.0, (2, 4)); b1 = np.zeros((1, 4))
    W2 = rng.uniform(-1.0, 1.0, (4, 1)); b2 = np.zeros((1, 1))
    losses = []
    
    for epoch in range(epochs):
        z1 = X @ W1 + b1; a1 = sigmoid(z1)
        z2 = a1 @ W2 + b2; a2 = sigmoid(z2)
        loss = 0.5 * np.mean((y - a2)**2)
        losses.append(loss)
        
        delta2 = (a2 - y) * sigmoid_deriv(a2)
        delta1 = (delta2 @ W2.T) * sigmoid_deriv(a1)
        
        W2 -= lr * (a1.T @ delta2) / len(X)
        b2 -= lr * np.sum(delta2, axis=0, keepdims=True) / len(X)
        W1 -= lr * (X.T @ delta1) / len(X)
        b1 -= lr * np.sum(delta1, axis=0, keepdims=True) / len(X)
        
    return (W1, b1, W2, b2), losses, a2

lrs = [0.1, 0.5, 2.0]
models = {}

for lr in lrs:
    weights, loss_history, preds = train_mlp(lr)
    models[lr] = (weights, loss_history, preds)
    print(f"Learning Rate: {lr:3.1f} -> Final Loss: {loss_history[-1]:.6f}")
    print(f"  Predictions: {preds.flatten().round(4)}")

# Plot Figure 6.1: Loss Curves
fig, ax = plt.subplots(figsize=(6.5, 3.6), dpi=150)
for lr in lrs:
    ax.plot(models[lr][1], label=f'$\eta = {lr}$')
ax.set_yscale('log')
ax.set_xlabel('Training Epoch')
ax.set_ylabel('Mean Squared Error Loss (log scale)')
ax.set_title('Convergence of 2-4-1 MLP on XOR for Different Learning Rates', fontsize=11)
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('figs/fig6_1_loss_curves.png')
plt.show()

# Plot Figure 6.2: Nonlinear Decision Boundary (for lr = 2.0)
best_w = models[2.0][0]
W1, b1, W2, b2 = best_w

xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
z1 = grid @ W1 + b1; a1 = sigmoid(z1)
z2 = a1 @ W2 + b2; a2 = sigmoid(z2)
zz = a2.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(5.2, 4.4), dpi=150)
contour = ax.contourf(xx, yy, zz, levels=20, cmap='RdBu_r', alpha=0.7)
plt.colorbar(contour, ax=ax, label='Network Output $\hat{y}$')
ax.contour(xx, yy, zz, levels=[0.5], colors='black', linewidths=2.0)

ax.scatter([0, 1], [1, 0], color='red', edgecolor='black', s=100, label='Target = 1')
ax.scatter([0, 1], [0, 1], color='blue', edgecolor='black', s=100, label='Target = 0')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Nonlinear Decision Boundary for XOR ($\eta = 2.0$)', fontsize=11)
ax.legend(loc='lower left')
plt.tight_layout()
plt.savefig('figs/fig6_2_decision_boundary.png')
plt.show()
